# GRU Seq2Seq Forecasting for M5

This notebook builds a lightweight GRU-based sequence forecasting model for the M5 Forecasting Accuracy competition.

The goal is to generate a valid full Kaggle submission using a sequence-based model.

Model idea:

- Input: past 90 days of sales
- Output: next 28 days of sales
- Transform: log1p(sales)
- Model: GRU sequence forecasting model
- Submission: full M5 validation and evaluation rows


In [1]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

PyTorch version: 2.10.0+cu128
CUDA available: True
Device: cuda


## 1. Configuration

We define the main settings for the lightweight GRU sequence model.

To keep the first version computationally manageable, we use only historical sales values as input.

In [7]:
# =========================
# Configuration
# =========================

DATA_DIR = "/kaggle/input/competitions/m5-forecasting-accuracy"

INPUT_LEN = 90
HORIZON = 28

# Number of recent training windows per item-store series
N_WINDOWS = 6
STEP = 28

BATCH_SIZE = 512
EPOCHS = 8
LR = 1e-3

HIDDEN_SIZE = 64
NUM_LAYERS = 1
DROPOUT = 0.1

RANDOM_STATE = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

## 2. Load M5 Data

We load the M5 sales data and the sample submission file.

The sales file contains one row per item-store series, with daily sales columns from `d_1` to `d_1913`.

In [8]:
# =========================
# Load data
# =========================

sales_path = os.path.join(DATA_DIR, "sales_train_validation.csv")
sample_path = os.path.join(DATA_DIR, "sample_submission.csv")

sales = pd.read_csv(sales_path)
sample_submission = pd.read_csv(sample_path)

print("sales shape:", sales.shape)
print("sample_submission shape:", sample_submission.shape)

sales.head()

sales shape: (30490, 1919)
sample_submission shape: (60980, 29)


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


In [5]:
import os

print(os.listdir("/kaggle/input"))

['competitions']


In [6]:
print(os.listdir("/kaggle/input/competitions"))

['m5-forecasting-accuracy']


In [10]:
# Identify daily sales columns
day_cols = [col for col in sales.columns if col.startswith("d_")]

print("Number of day columns:", len(day_cols))
print("First 5 day columns:", day_cols[:5])
print("Last 5 day columns:", day_cols[-5:])

sales[["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]].head()

Number of day columns: 1913
First 5 day columns: ['d_1', 'd_2', 'd_3', 'd_4', 'd_5']
Last 5 day columns: ['d_1909', 'd_1910', 'd_1911', 'd_1912', 'd_1913']


,id,item_id,dept_id,cat_id,store_id,state_id
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA


## 3. Prepare Sales Sequences

We extract the daily sales columns and convert them into a NumPy matrix.

Each row represents one item-store time series, and each column represents one day.

In [11]:
# =========================
# Prepare sales matrix
# =========================

sales_values = sales[day_cols].values.astype(np.float32)

print("sales_values shape:", sales_values.shape)
print("Min sales:", sales_values.min())
print("Max sales:", sales_values.max())
print("Mean sales:", sales_values.mean())

sales_values shape: (30490, 1913)
Min sales: 0.0
Max sales: 763.0
Mean sales: 1.1263218


## 4. Apply Log Transformation

Sales values are highly skewed and many item-store series contain zeros.

We apply `log1p(sales)` to stabilize the scale:

`log1p(x) = log(1 + x)`

After prediction, we will use `expm1()` to convert the forecasts back to the original sales scale.

In [12]:
# =========================
# Log transform
# =========================

sales_values_log = np.log1p(sales_values)

print("Original max:", sales_values.max())
print("Log-transformed max:", sales_values_log.max())
print("Original sample:", sales_values[0, -10:])
print("Log sample:", sales_values_log[0, -10:])

Original max: 763.0
Log-transformed max: 6.638568
Original sample: [1. 3. 0. 1. 1. 1. 3. 0. 1. 1.]
Log sample: [0.6931472 1.3862944 0.        0.6931472 0.6931472 0.6931472 1.3862944
 0.        0.6931472 0.6931472]


## 5. Create Recent Sliding Windows

To train the GRU model, we convert each item-store sales sequence into supervised learning samples.

Each sample contains:

- `X`: past 90 days of sales
- `y`: next 28 days of sales

To keep training manageable, we only use a limited number of recent windows from each item-store series.

In [13]:
# =========================
# Create recent sliding windows
# =========================

def make_recent_windows(values, input_len=90, horizon=28, n_windows=6, step=28):
    """
    Convert sales sequences into recent training windows.

    Parameters
    ----------
    values : np.ndarray
        Sales matrix with shape (n_series, n_days).
        Each row is one item-store time series.
    input_len : int
        Number of past days used as model input.
    horizon : int
        Number of future days to predict.
    n_windows : int
        Number of recent windows to create per item-store series.
    step : int
        Distance between window start positions.

    Returns
    -------
    X : np.ndarray
        Input sequences with shape (n_samples, input_len, 1).
    y : np.ndarray
        Forecast targets with shape (n_samples, horizon).
    """
    n_series, n_days = values.shape

    X_list = []
    y_list = []

    for i in range(n_windows):
        offset = i * step

        # Start from the most recent possible training window and move backward
        start = n_days - input_len - horizon - offset
        end_x = start + input_len
        end_y = end_x + horizon

        if start < 0:
            continue

        X_window = values[:, start:end_x]
        y_window = values[:, end_x:end_y]

        X_list.append(X_window)
        y_list.append(y_window)

        print(
            f"Window {i+1}: "
            f"X days index {start}:{end_x}, "
            f"y days index {end_x}:{end_y}"
        )

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)

    # Add feature dimension for GRU: (samples, timesteps, features)
    X = X[:, :, None]

    return X.astype(np.float32), y.astype(np.float32)


X, y = make_recent_windows(
    sales_values_log,
    input_len=INPUT_LEN,
    horizon=HORIZON,
    n_windows=N_WINDOWS,
    step=STEP
)

print("X shape:", X.shape)
print("y shape:", y.shape)

Window 1: X days index 1795:1885, y days index 1885:1913
Window 2: X days index 1767:1857, y days index 1857:1885
Window 3: X days index 1739:1829, y days index 1829:1857
Window 4: X days index 1711:1801, y days index 1801:1829
Window 5: X days index 1683:1773, y days index 1773:1801
Window 6: X days index 1655:1745, y days index 1745:1773
X shape: (182940, 90, 1)
y shape: (182940, 28)


## 6. Train / Validation Split

We split the generated sequence windows into training and validation sets.

The validation set is used to monitor whether the GRU model is learning useful patterns instead of only memorizing the training data.

In [14]:
# =========================
# Train / validation split
# =========================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.1,
    random_state=RANDOM_STATE
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

X_train shape: (164646, 90, 1)
y_train shape: (164646, 28)
X_val shape: (18294, 90, 1)
y_val shape: (18294, 28)


## 7. Create PyTorch DataLoaders

PyTorch models are usually trained in mini-batches.

We convert the NumPy arrays into tensors and use `DataLoader` to feed batches of data into the GRU model during training.

In [16]:
# =========================
# Create PyTorch datasets and dataloaders
# =========================

train_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)

val_ds = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32)
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Number of training batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))

xb, yb = next(iter(train_loader))
print("One X batch shape:", xb.shape)
print("One y batch shape:", yb.shape)

Number of training batches: 322
Number of validation batches: 36
One X batch shape: torch.Size([512, 90, 1])
One y batch shape: torch.Size([512, 28])


## 8. Define GRU Forecasting Model

The GRU model reads a 90-day sales sequence and summarizes it into a hidden representation.

We then use a fully connected layer to convert that representation into a 28-day forecast.

In [24]:
# =========================
# Define GRU forecasting model
# =========================

class GRUForecaster(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1, horizon=28, dropout=0.1):
        super().__init__()
        
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, horizon),
            nn.Softplus()
        )
        
    def forward(self, x):
        # x shape: (batch_size, input_len, input_size)
        output, hidden = self.gru(x)
        
        # Use the hidden representation from the final time step
        last_hidden = output[:, -1, :]
        
        # Convert hidden representation into 28-day forecast
        forecast = self.fc(last_hidden)
        
        return forecast


model = GRUForecaster(
    input_size=1,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    horizon=HORIZON,
    dropout=DROPOUT
).to(DEVICE)

model

GRUForecaster(
  (gru): GRU(1, 64, batch_first=True)
  (fc): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64, out_features=28, bias=True)
    (4): Softplus(beta=1.0, threshold=20.0)
  )
)

In [25]:
# Test one forward pass
xb, yb = next(iter(train_loader))
xb = xb.to(DEVICE)

with torch.no_grad():
    test_preds = model(xb)

print("Input batch shape:", xb.shape)
print("Prediction shape:", test_preds.shape)
print("Target shape:", yb.shape)

Input batch shape: torch.Size([512, 90, 1])
Prediction shape: torch.Size([512, 28])
Target shape: torch.Size([512, 28])


## 9. Train the GRU Model

We train the GRU model using Mean Squared Error on the log-transformed sales values.

The model learns to map a 90-day historical sales sequence to a 28-day future sales sequence.

In [26]:
# =========================
# Loss function and optimizer
# =========================

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: MSELoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [27]:
# =========================
# Training and evaluation functions
# =========================

def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        
        optimizer.zero_grad()
        
        preds = model(xb)
        loss = criterion(preds, yb)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * xb.size(0)
    
    avg_loss = total_loss / len(train_loader.dataset)
    return avg_loss


def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            
            preds = model(xb)
            loss = criterion(preds, yb)
            
            total_loss += loss.item() * xb.size(0)
    
    avg_loss = total_loss / len(val_loader.dataset)
    return avg_loss

In [28]:
# =========================
# Train model
# =========================

train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE
    )
    
    val_loss = evaluate(
        model=model,
        val_loader=val_loader,
        criterion=criterion,
        device=DEVICE
    )
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"train_loss: {train_loss:.5f} | "
        f"val_loss: {val_loss:.5f}"
    )

Epoch 1/8 | train_loss: 0.28247 | val_loss: 0.25077
Epoch 2/8 | train_loss: 0.25058 | val_loss: 0.24847
Epoch 3/8 | train_loss: 0.24919 | val_loss: 0.24791
Epoch 4/8 | train_loss: 0.24840 | val_loss: 0.24766
Epoch 5/8 | train_loss: 0.24791 | val_loss: 0.24745
Epoch 6/8 | train_loss: 0.24748 | val_loss: 0.24696
Epoch 7/8 | train_loss: 0.24721 | val_loss: 0.24720
Epoch 8/8 | train_loss: 0.24701 | val_loss: 0.24650


## 10. Generate Forecasts for All Item-Store Series

After training, we use the most recent 90 days of sales for each item-store series to predict the next 28 days.

The model outputs predictions in log scale, so we convert them back to the original sales scale using `expm1()`.

In [29]:
# =========================
# Generate forecasts for all item-store series
# =========================

last_input = sales_values_log[:, -INPUT_LEN:]
last_input = last_input[:, :, None].astype(np.float32)

print("last_input shape:", last_input.shape)

last_input_tensor = torch.tensor(last_input, dtype=torch.float32)

preds_list = []

model.eval()
with torch.no_grad():
    pred_loader = DataLoader(
        last_input_tensor,
        batch_size=BATCH_SIZE,
        shuffle=False
    )
    
    for xb in pred_loader:
        xb = xb.to(DEVICE)
        preds = model(xb)
        preds_list.append(preds.cpu().numpy())

preds_log = np.concatenate(preds_list, axis=0)

print("preds_log shape:", preds_log.shape)
print("preds_log min:", preds_log.min())
print("preds_log max:", preds_log.max())

last_input shape: (30490, 90, 1)
preds_log shape: (30490, 28)
preds_log min: 0.032729052
preds_log max: 4.693163


In [30]:
# =========================
# Convert predictions back to sales scale
# =========================

preds = np.expm1(preds_log)

print("preds shape:", preds.shape)
print("preds min:", preds.min())
print("preds max:", preds.max())
print("preds mean:", preds.mean())
print("Number of negative predictions:", (preds < 0).sum())

preds[:3]

preds shape: (30490, 28)
preds min: 0.03327054
preds max: 108.19802
preds mean: 1.013952
Number of negative predictions: 0


array([[0.7279761 , 0.65254176, 0.6490465 , 0.65519375, 0.7339889 ,
        0.97672427, 0.9652088 , 0.706733  , 0.648348  , 0.64471376,
        0.65497786, 0.7311169 , 0.9099589 , 0.882566  , 0.7207683 ,
        0.6564047 , 0.6331695 , 0.55076283, 0.5104571 , 0.7808374 ,
        0.8249714 , 0.62627697, 0.61139196, 0.60459757, 0.63979566,
        0.6937516 , 0.8775189 , 0.9053843 ],
       [0.09960476, 0.08747979, 0.09634177, 0.0963228 , 0.12053491,
        0.15438984, 0.1462597 , 0.11175542, 0.10318216, 0.10807617,
        0.11002615, 0.12829797, 0.15806624, 0.14861315, 0.12404925,
        0.11737913, 0.11703829, 0.10655813, 0.09115955, 0.13835682,
        0.14689085, 0.10854729, 0.10927299, 0.11325758, 0.12549797,
        0.13653551, 0.16733202, 0.17020628],
       [0.4681262 , 0.4171508 , 0.41639304, 0.41585267, 0.47392192,
        0.6261826 , 0.6108142 , 0.4510869 , 0.41579726, 0.41643175,
        0.41829294, 0.47183153, 0.5841575 , 0.5638607 , 0.46472546,
        0.4250746 , 0.4152

## 11. Build Kaggle Submission

We convert the GRU predictions into the required Kaggle submission format.

The M5 submission file contains both validation and evaluation rows. Since this notebook uses `sales_train_validation.csv`, we generate forecasts for the validation IDs and copy the same forecast structure to the corresponding evaluation IDs for a valid full submission.

In [31]:
# =========================
# Build submission file
# =========================

forecast_cols = [f"F{i}" for i in range(1, HORIZON + 1)]

pred_df = pd.DataFrame(preds, columns=forecast_cols)
pred_df.insert(0, "id", sales["id"].values)

print("pred_df shape:", pred_df.shape)
pred_df.head()

pred_df shape: (30490, 29)


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_validation,0.727976,0.652542,0.649046,0.655194,0.733989,0.976724,0.965209,0.706733,0.648348,...,0.510457,0.780837,0.824971,0.626277,0.611392,0.604598,0.639796,0.693752,0.877519,0.905384
1,HOBBIES_1_002_CA_1_validation,0.099605,0.087480,0.096342,0.096323,0.120535,0.154390,0.146260,0.111755,0.103182,...,0.091160,0.138357,0.146891,0.108547,0.109273,0.113258,0.125498,0.136536,0.167332,0.170206
2,HOBBIES_1_003_CA_1_validation,0.468126,0.417151,0.416393,0.415853,0.473922,0.626183,0.610814,0.451087,0.415797,...,0.328962,0.498313,0.528607,0.398448,0.391043,0.385234,0.417186,0.450731,0.567580,0.582286
3,HOBBIES_1_004_CA_1_validation,1.553780,1.346622,1.377767,1.350370,1.526417,2.105657,2.038976,1.476571,1.352375,...,0.993547,1.647312,1.706157,1.278375,1.240650,1.245151,1.284456,1.429619,1.817640,1.897800
4,HOBBIES_1_005_CA_1_validation,0.992301,0.848159,0.861528,0.852056,0.975735,1.349916,1.314674,0.928146,0.856184,...,0.611564,1.033062,1.098579,0.797689,0.768615,0.782980,0.824969,0.907885,1.173241,1.227794


In [ ]:
# Align predictions with sample_submission order
submission = sample_submission[["id"]].merge(
    all_pred_df,
    on="id",
    how="left"
)

print("submission shape:", submission.shape)
print("Number of missing values:", submission.isna().sum().sum())

submission.head()

In [32]:
# Copy validation predictions to evaluation IDs
eval_pred_df = pred_df.copy()
eval_pred_df["id"] = eval_pred_df["id"].str.replace("_validation", "_evaluation", regex=False)

# Combine validation and evaluation predictions
all_pred_df = pd.concat([pred_df, eval_pred_df], axis=0, ignore_index=True)

print("all_pred_df shape:", all_pred_df.shape)
all_pred_df.head()

all_pred_df shape: (60980, 29)


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_validation,0.727976,0.652542,0.649046,0.655194,0.733989,0.976724,0.965209,0.706733,0.648348,...,0.510457,0.780837,0.824971,0.626277,0.611392,0.604598,0.639796,0.693752,0.877519,0.905384
1,HOBBIES_1_002_CA_1_validation,0.099605,0.087480,0.096342,0.096323,0.120535,0.154390,0.146260,0.111755,0.103182,...,0.091160,0.138357,0.146891,0.108547,0.109273,0.113258,0.125498,0.136536,0.167332,0.170206
2,HOBBIES_1_003_CA_1_validation,0.468126,0.417151,0.416393,0.415853,0.473922,0.626183,0.610814,0.451087,0.415797,...,0.328962,0.498313,0.528607,0.398448,0.391043,0.385234,0.417186,0.450731,0.567580,0.582286
3,HOBBIES_1_004_CA_1_validation,1.553780,1.346622,1.377767,1.350370,1.526417,2.105657,2.038976,1.476571,1.352375,...,0.993547,1.647312,1.706157,1.278375,1.240650,1.245151,1.284456,1.429619,1.817640,1.897800
4,HOBBIES_1_005_CA_1_validation,0.992301,0.848159,0.861528,0.852056,0.975735,1.349916,1.314674,0.928146,0.856184,...,0.611564,1.033062,1.098579,0.797689,0.768615,0.782980,0.824969,0.907885,1.173241,1.227794


In [33]:
# Align predictions with sample_submission order
submission = sample_submission[["id"]].merge(
    all_pred_df,
    on="id",
    how="left"
)

print("submission shape:", submission.shape)
print("Number of missing values:", submission.isna().sum().sum())

submission.head()

submission shape: (60980, 29)
Number of missing values: 0


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_validation,0.727976,0.652542,0.649046,0.655194,0.733989,0.976724,0.965209,0.706733,0.648348,...,0.510457,0.780837,0.824971,0.626277,0.611392,0.604598,0.639796,0.693752,0.877519,0.905384
1,HOBBIES_1_002_CA_1_validation,0.099605,0.087480,0.096342,0.096323,0.120535,0.154390,0.146260,0.111755,0.103182,...,0.091160,0.138357,0.146891,0.108547,0.109273,0.113258,0.125498,0.136536,0.167332,0.170206
2,HOBBIES_1_003_CA_1_validation,0.468126,0.417151,0.416393,0.415853,0.473922,0.626183,0.610814,0.451087,0.415797,...,0.328962,0.498313,0.528607,0.398448,0.391043,0.385234,0.417186,0.450731,0.567580,0.582286
3,HOBBIES_1_004_CA_1_validation,1.553780,1.346622,1.377767,1.350370,1.526417,2.105657,2.038976,1.476571,1.352375,...,0.993547,1.647312,1.706157,1.278375,1.240650,1.245151,1.284456,1.429619,1.817640,1.897800
4,HOBBIES_1_005_CA_1_validation,0.992301,0.848159,0.861528,0.852056,0.975735,1.349916,1.314674,0.928146,0.856184,...,0.611564,1.033062,1.098579,0.797689,0.768615,0.782980,0.824969,0.907885,1.173241,1.227794


In [34]:
output_path = "/kaggle/working/gru_seq2seq_softplus_submission_v1.csv"
submission.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /kaggle/working/gru_seq2seq_softplus_submission_v1.csv


In [38]:
print("Prediction summary")
print("preds shape:", preds.shape)
print("min:", preds.min())
print("max:", preds.max())
print("mean:", preds.mean())
print("median:", np.median(preds))

print("\nRecent actual sales summary")
recent_actual = sales_values[:, -28:]
print("min:", recent_actual.min())
print("max:", recent_actual.max())
print("mean:", recent_actual.mean())
print("median:", np.median(recent_actual))

Prediction summary
preds shape: (30490, 28)
min: 0.03327054
max: 108.19802
mean: 1.013952
median: 0.42339596

Recent actual sales summary
min: 0.0
max: 204.0
mean: 1.3864335
median: 0.0


In [39]:
recent_90_sum = sales_values[:, -90:].sum(axis=1)
recent_28_sum = sales_values[:, -28:].sum(axis=1)
pred_28_sum = preds.sum(axis=1)

zero_recent_mask = recent_90_sum == 0

print("Number of series with zero sales in last 90 days:", zero_recent_mask.sum())
print("Share of zero-recent-sales series:", zero_recent_mask.mean())

print("\nFor zero-recent-sales series:")
print("Average predicted 28-day total:", pred_28_sum[zero_recent_mask].mean())
print("Median predicted 28-day total:", np.median(pred_28_sum[zero_recent_mask]))
print("Max predicted 28-day total:", pred_28_sum[zero_recent_mask].max())

print("\nFor all series:")
print("Average predicted 28-day total:", pred_28_sum.mean())
print("Average actual recent 28-day total:", recent_28_sum.mean())

Number of series with zero sales in last 90 days: 390
Share of zero-recent-sales series: 0.012791079042308954

For zero-recent-sales series:
Average predicted 28-day total: 3.4041517
Median predicted 28-day total: 3.4041512
Max predicted 28-day total: 3.4041512

For all series:
Average predicted 28-day total: 28.390646
Average actual recent 28-day total: 38.820137
